# 02 – Data Understanding: Análisis Exploratorio de Datos (EDA)

**Proyecto:** Encuesta Permanente de Empleo Nacional (EPEN)  
**Objetivo:** Explorar el dataset para entender la distribución de las variables, detectar outliers, relaciones entre variables y obtener insights iniciales.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

# ─── Cargar datos ─────────────────────────────────────────────────────────────
RAW_PATH = os.path.join('..', 'data', 'raw', 'epen_snapshot.csv')
try:
    df = pd.read_csv(RAW_PATH)
except FileNotFoundError:
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'edad': np.random.randint(14, 70, n),
        'sexo': np.random.choice(['Hombre', 'Mujer'], n),
        'nivel_educativo': np.random.choice(
            ['Sin instrucción', 'Primaria', 'Secundaria', 'Preparatoria', 'Universidad', 'Posgrado'], n
        ),
        'estado_civil': np.random.choice(['Soltero', 'Casado', 'Unión libre', 'Divorciado', 'Viudo'], n),
        'ingreso_mensual': np.random.exponential(8000, n).round(2),
        'horas_trabajadas': np.random.randint(0, 60, n),
        'tipo_empleo': np.random.choice(['Formal', 'Informal', np.nan], n, p=[0.45, 0.45, 0.10]),
        'sector': np.random.choice(['Agricultura', 'Industria', 'Comercio', 'Servicios', 'Gobierno'], n),
        'condicion_actividad': np.random.choice(
            ['Ocupado', 'Desocupado', 'No PEA'], n, p=[0.60, 0.10, 0.30]
        ),
    })

df['target_desocupado'] = (df['condicion_actividad'] == 'Desocupado').astype(int)
print(f'Dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas')
df.head()

## 1. Variables numéricas

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
print('Variables numéricas:', num_cols)
df[num_cols].describe().T

In [ ]:
# Histogramas de variables numéricas
num_plot = [c for c in num_cols if c != 'target_desocupado']
fig, axes = plt.subplots(1, len(num_plot), figsize=(5 * len(num_plot), 4))
if len(num_plot) == 1:
    axes = [axes]
for ax, col in zip(axes, num_plot):
    ax.hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white')
    ax.set_title(col)
    ax.set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()

## 2. Variables categóricas

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print('Variables categóricas:', cat_cols)

n_cols_grid = 2
n_rows_grid = int(np.ceil(len(cat_cols) / n_cols_grid))
fig, axes = plt.subplots(n_rows_grid, n_cols_grid, figsize=(14, 4 * n_rows_grid))
axes = axes.flatten()
for ax, col in zip(axes, cat_cols):
    vc = df[col].value_counts()
    ax.bar(vc.index, vc.values, color='steelblue', edgecolor='black')
    ax.set_title(col)
    ax.set_ylabel('Frecuencia')
    ax.tick_params(axis='x', rotation=30)
for ax in axes[len(cat_cols):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

## 3. Correlación entre variables numéricas

In [ ]:
corr_matrix = df[num_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, square=True)
plt.title('Matriz de correlación – Variables numéricas')
plt.tight_layout()
plt.show()

## 4. Relación con el target

In [ ]:
# Distribución de edad por condición de actividad
plt.figure(figsize=(10, 4))
for cond, grp in df.groupby('condicion_actividad'):
    grp['edad'].hist(bins=20, alpha=0.6, label=cond)
plt.title('Distribución de edad por condición de actividad')
plt.xlabel('Edad')
plt.ylabel('Frecuencia')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Tasa de desocupación por nivel educativo
tasa_ed = df.groupby('nivel_educativo')['target_desocupado'].mean().sort_values(ascending=False)
tasa_ed.plot(kind='bar', color='coral', edgecolor='black', figsize=(10, 4))
plt.title('Tasa de desocupación por nivel educativo')
plt.ylabel('Tasa de desocupación')
plt.xlabel('')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 5. Valores faltantes

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
missing_df = pd.DataFrame({'Nulos': missing, 'Nulos_%': missing_pct.round(2)})
print(missing_df[missing_df['Nulos'] > 0])

## 6. Conclusiones del EDA

- Documentar aquí los hallazgos más importantes.
- Ejemplo: La variable `ingreso_mensual` presenta distribución sesgada a la derecha; considerar transformación logarítmica.
- La variable `tipo_empleo` tiene ~10% de valores nulos que corresponden a personas fuera de la PEA.